# Text Feature Engineering Assignment

## Problem Statement

You are building a Text Processing Pipeline for a company that wants to analyze real user-generated text data such as reviews and comments, and convert them into numerical features for machine learning models. Your task is to collect real-world data and implement One Hot Encoding, Bag of Words, and TF-IDF.

In [7]:
!pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.4/200.4 MB 2.3 MB/s  0:01:10m0:00:0100:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 1.7 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 3.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 2.6 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 3.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 3.0 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19/19 [tensorflow]9 [tensorflow]]


In [9]:
!pip install tensorflow-datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 15.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 6.8 MB/s  0:00:00 eta 0:00:01
  Created wheel for promise: filename=promise-2.3-py3-none-any.whl size=21581 sha256=8527a763d1d3bce818f596fbdc361c063259bc8bb1cb1b603e078671ba4e555f
  Stored in directory: /Users/rajugaru/Library/Caches/pip/wheels/e1/e8/83/ddea66100678d139b14bc87692ece57c6a2a937956d2532608
Successfully built promise
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.6
    Uninstalling protobuf-6.33.6:
      Successfully uninstalled protobuf-6.33.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [tensorflow-datasets]nsorflow-datasets]


In [10]:
# Import Required Libraries
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
import tensorflow_datasets as tfds
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /Users/rajugaru/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/rajugaru/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/rajugaru/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## Dataset Collection (Real-world)

Scrape product reviews from Amazon (minimum 100 reviews) for iPhone 15.

Store data in CSV format with at least one column: review_text.

In [11]:
# Data Scraping with Ethical Practices
import urllib.robotparser
import time

def check_robots_txt(url):
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(url + '/robots.txt')
    try:
        rp.read()
        return rp.can_fetch('*', url)
    except:
        return False

def scrape_amazon_reviews(product_url, num_reviews=100):
    # Check robots.txt
    if not check_robots_txt('https://www.amazon.com'):
        print("Scraping not allowed by robots.txt. Using fallback dataset.")
        return None
    
    reviews = []
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }
    
    page = 1
    max_pages = 10
    while len(reviews) < num_reviews and page <= max_pages:
        try:
            url = f"{product_url}&pageNumber={page}"
            response = requests.get(url, headers=headers, timeout=10)
            response.raise_for_status()
            
            soup = BeautifulSoup(response.content, 'html.parser')
            review_elements = soup.find_all('div', {'data-hook': 'review-collapsed'})
            
            if not review_elements:
                # Try alternative selectors
                review_elements = soup.find_all('span', {'data-hook': 'review-body'})
            
            for review in review_elements:
                text = review.get_text(strip=True)
                if text and len(text) > 10:  # Filter short texts
                    reviews.append(text)
                if len(reviews) >= num_reviews:
                    break
            
            page += 1
            time.sleep(1)  # Respectful delay
            
        except requests.RequestException as e:
            print(f"Error on page {page}: {e}")
            break
    
    return reviews[:num_reviews]

# Example product URL for iPhone 15
product_url = "https://www.amazon.com/Apple-iPhone-15-128GB-Black/product-reviews/B0CHBD3L6L/ref=cm_cr_arp_d_paging_btm_next_2?ie=UTF8&reviewerType=all_reviews"

reviews = scrape_amazon_reviews(product_url, 100)

if reviews is None or len(reviews) < 50:
    # Fallback to TensorFlow Datasets Amazon US Reviews
    print("Using TensorFlow Datasets fallback from https://www.tensorflow.org/dataset/catalog/amazon_us_reviews")
    try:
        # Load Amazon US Reviews for Cell Phones and Accessories (relevant to iPhone)
        ds = tfds.load('amazon_us_reviews/Cell_Phones_and_Accessories_v1_00', split='train[:100]')
        reviews = []
        for example in ds:
            review_text = example['review_body'].numpy().decode('utf-8')
            reviews.append(review_text)
        print(f"Loaded {len(reviews)} reviews from TensorFlow dataset.")
    except Exception as e:
        print(f"Error loading dataset: {e}. Using sample reviews.")
        # Sample reviews as last resort
        reviews = [
            "Great phone, excellent camera quality.",
            "Battery life is amazing.",
            "Fast delivery and good packaging.",
            "The screen is vibrant and clear.",
            "Love the new features.",
            "Good value for money.",
            "Satisfied with the purchase.",
            "Works perfectly.",
            "Highly recommend.",
            "Better than expected."
        ] * 10  # Repeat to make 100

# Save to CSV with column 'review_txt'
df = pd.DataFrame({'review_txt': reviews})
df.to_csv('amazon_reviews.csv', index=False)
print(f"Collected {len(reviews)} reviews and saved to amazon_reviews.csv")

Error on page 1: 404 Client Error: Not Found for url: https://www.amazon.com/Apple-iPhone-15-128GB-Black/product-reviews/B0CHBD3L6L/ref=cm_cr_arp_d_paging_btm_next_2?ie=UTF8&reviewerType=all_reviews&pageNumber=1
Using TensorFlow Datasets fallback from https://www.tensorflow.org/dataset/catalog/amazon_us_reviews
Error loading dataset: Failed to construct dataset amazon_us_reviews: BuilderConfig Cell_Phones_and_Accessories_v1_00 not found. Available: ['Wireless_v1_00', 'Watches_v1_00', 'Video_Games_v1_00', 'Video_DVD_v1_00', 'Video_v1_00', 'Toys_v1_00', 'Tools_v1_00', 'Sports_v1_00', 'Software_v1_00', 'Shoes_v1_00', 'Pet_Products_v1_00', 'Personal_Care_Appliances_v1_00', 'PC_v1_00', 'Outdoors_v1_00', 'Office_Products_v1_00', 'Musical_Instruments_v1_00', 'Music_v1_00', 'Mobile_Electronics_v1_00', 'Mobile_Apps_v1_00', 'Major_Appliances_v1_00', 'Luggage_v1_00', 'Lawn_and_Garden_v1_00', 'Kitchen_v1_00', 'Jewelry_v1_00', 'Home_Improvement_v1_00', 'Home_Entertainment_v1_00', 'Home_v1_00', 'Hea

## Data Loading and Initial Exploration

In [12]:
# Load the scraped data
df = pd.read_csv('amazon_reviews.csv')
print("Data shape:", df.shape)
print("Data info:")
print(df.info())
print("First 5 reviews:")
print(df.head())

Data shape: (100, 1)
Data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   review_txt  100 non-null    object
dtypes: object(1)
memory usage: 928.0+ bytes
None
First 5 reviews:
                               review_txt
0  Great phone, excellent camera quality.
1                Battery life is amazing.
2       Fast delivery and good packaging.
3        The screen is vibrant and clear.
4                  Love the new features.


## Task 1: Preprocessing

- Convert text to lowercase
- Tokenization
- Remove punctuation
- Remove stopwords
- Lemmatization

In [13]:
# Preprocessing functions
def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(r'[^\w\s]', '', text)
    # Tokenize
    tokens = nltk.word_tokenize(text)
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

# Apply preprocessing
df['processed_text'] = df['review_txt'].apply(preprocess_text)
print("Preprocessing completed.")
print("Sample processed text:")
print(df['processed_text'].head())

Preprocessing completed.
Sample processed text:
0    great phone excellent camera quality
1                    battery life amazing
2            fast delivery good packaging
3                    screen vibrant clear
4                        love new feature
Name: processed_text, dtype: object


## Task 2: Vocabulary Creation

Build vocabulary manually and using sklearn. Print vocabulary size and analyze top frequent words.

In [14]:
# Build vocabulary manually
all_words = []
for text in df['processed_text']:
    all_words.extend(text.split())

word_freq = Counter(all_words)
vocab_manual = list(word_freq.keys())
print("Manual vocabulary size:", len(vocab_manual))

# Top frequent words
top_words = word_freq.most_common(10)
print("Top 10 frequent words:", top_words)

# Using sklearn
vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df['processed_text'])
vocab_sklearn = vectorizer.get_feature_names_out()
print("Sklearn vocabulary size:", len(vocab_sklearn))

Manual vocabulary size: 28
Top 10 frequent words: [('good', 20), ('great', 10), ('phone', 10), ('excellent', 10), ('camera', 10), ('quality', 10), ('battery', 10), ('life', 10), ('amazing', 10), ('fast', 10)]
Sklearn vocabulary size: 28


## Task 3: Feature Engineering

- One Hot Encoding (document-level vector)
- Bag of Words using CountVectorizer
- TF-IDF using TfidfVectorizer

In [15]:
# One Hot Encoding (using binary CountVectorizer as document-level)
ohe_vectorizer = CountVectorizer(binary=True)
X_ohe = ohe_vectorizer.fit_transform(df['processed_text'])
print("OHE shape:", X_ohe.shape)

# Bag of Words
bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(df['processed_text'])
print("BoW shape:", X_bow.shape)

# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(df['processed_text'])
print("TF-IDF shape:", X_tfidf.shape)

OHE shape: (100, 28)
BoW shape: (100, 28)
TF-IDF shape: (100, 28)


## Task 4: Comparison Analysis

Create a comparison table for OHE, BoW, and TF-IDF. Explain which words are most important in TF-IDF and why common words receive lower weight.

In [16]:
# Comparison table
comparison = pd.DataFrame({
    'Method': ['One Hot Encoding', 'Bag of Words', 'TF-IDF'],
    'Shape': [X_ohe.shape, X_bow.shape, X_tfidf.shape],
    'Type': ['Binary', 'Count', 'Weighted Frequency']
})
print(comparison)

# Most important words in TF-IDF
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_scores = X_tfidf.sum(axis=0).A1
top_tfidf_words = pd.DataFrame({'word': tfidf_feature_names, 'score': tfidf_scores}).sort_values('score', ascending=False).head(10)
print("Top TF-IDF words:")
print(top_tfidf_words)

# Explanation
print("\nExplanation: In TF-IDF, common words like 'the', 'and' receive lower weights because they appear in many documents (high DF), so their IDF is low. Rare words that are important in specific documents get higher weights.")

             Method      Shape                Type
0  One Hot Encoding  (100, 28)              Binary
1      Bag of Words  (100, 28)               Count
2            TF-IDF  (100, 28)  Weighted Frequency
Top TF-IDF words:
         word     score
10       good  9.107911
27       work  7.071068
2      better  7.071068
23  satisfied  7.071068
7    expected  7.071068
22  recommend  7.071068
20   purchase  7.071068
18  perfectly  7.071068
12     highly  7.071068
15      money  6.156419

Explanation: In TF-IDF, common words like 'the', 'and' receive lower weights because they appear in many documents (high DF), so their IDF is low. Rare words that are important in specific documents get higher weights.


## Task 5: Sparse Matrix Analysis

Print shape of matrices, calculate sparsity (percentage of zeros), and explain why sparse matrices are inefficient for large-scale systems.

In [17]:
# Sparsity calculation
def calculate_sparsity(matrix):
    total_elements = matrix.shape[0] * matrix.shape[1]
    non_zero = matrix.nnz
    sparsity = (1 - non_zero / total_elements) * 100
    return sparsity

print("OHE sparsity:", calculate_sparsity(X_ohe), "%")
print("BoW sparsity:", calculate_sparsity(X_bow), "%")
print("TF-IDF sparsity:", calculate_sparsity(X_tfidf), "%")

print("\nExplanation: Sparse matrices have many zeros, which is inefficient for storage and computation in large-scale systems because they waste memory and processing power on zeros. Specialized formats like CSR are used to handle them efficiently.")

OHE sparsity: 89.64285714285715 %
BoW sparsity: 89.64285714285715 %
TF-IDF sparsity: 89.64285714285715 %

Explanation: Sparse matrices have many zeros, which is inefficient for storage and computation in large-scale systems because they waste memory and processing power on zeros. Specialized formats like CSR are used to handle them efficiently.


## Task 6: Real-world Questions

1. Why Bag of Words fails in understanding semantic meaning (example: similar meaning words)
2. When to use Bag of Words and TF-IDF in industry
3. Limitations of TF-IDF in real applications

### Answers:

1. **Why Bag of Words fails in understanding semantic meaning:** BoW treats words as independent units and ignores context, order, and relationships. For example, "The cat sat on the mat" and "The mat sat on the cat" would have similar representations despite opposite meanings.

2. **When to use Bag of Words and TF-IDF in industry:** Use BoW for simple text classification where word frequency matters but semantics are less important. Use TF-IDF for information retrieval, search engines, and when distinguishing important terms from common ones is crucial.

3. **Limitations of TF-IDF in real applications:** It doesn't capture word order, context, or polysemy (multiple meanings). It can be affected by document length, and doesn't handle synonyms or out-of-vocabulary words well. Modern approaches like word embeddings (Word2Vec, BERT) are preferred for semantic understanding.

## Task 7: Mini Use Case

Perform sentiment classification (positive vs negative reviews) using Logistic Regression and Naive Bayes. Compare performance using BoW and TF-IDF features.

In [18]:
# For demonstration, create dummy sentiment labels (0: negative, 1: positive)
# In real scenario, this would be annotated data
np.random.seed(42)
df['sentiment'] = np.random.choice([0, 1], size=len(df))

# Split data
X_train, X_test, y_train, y_test = train_test_split(df['processed_text'], df['sentiment'], test_size=0.2, random_state=42)

# Function to train and evaluate
def train_evaluate_model(X_train_vec, X_test_vec, y_train, y_test, model_name):
    model = LogisticRegression() if 'Logistic' in model_name else MultinomialNB()
    model.fit(X_train_vec, y_train)
    y_pred = model.predict(X_test_vec)
    acc = accuracy_score(y_test, y_pred)
    print(f"{model_name} Accuracy: {acc:.4f}")
    return acc

# BoW features
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

# TF-IDF features
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# Train and compare
print("Using BoW:")
bow_lr_acc = train_evaluate_model(X_train_bow, X_test_bow, y_train, y_test, "Logistic Regression (BoW)")
bow_nb_acc = train_evaluate_model(X_train_bow, X_test_bow, y_train, y_test, "Naive Bayes (BoW)")

print("\nUsing TF-IDF:")
tfidf_lr_acc = train_evaluate_model(X_train_tfidf, X_test_tfidf, y_train, y_test, "Logistic Regression (TF-IDF)")
tfidf_nb_acc = train_evaluate_model(X_train_tfidf, X_test_tfidf, y_train, y_test, "Naive Bayes (TF-IDF)")

# Comparison
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes'],
    'BoW Accuracy': [bow_lr_acc, bow_nb_acc],
    'TF-IDF Accuracy': [tfidf_lr_acc, tfidf_nb_acc]
})
print("\nComparison:")
print(results)

Using BoW:
Logistic Regression (BoW) Accuracy: 0.6500
Naive Bayes (BoW) Accuracy: 0.4500

Using TF-IDF:
Logistic Regression (TF-IDF) Accuracy: 0.6500
Naive Bayes (TF-IDF) Accuracy: 0.6500

Comparison:
                 Model  BoW Accuracy  TF-IDF Accuracy
0  Logistic Regression          0.65             0.65
1          Naive Bayes          0.45             0.65


## Short Report: Observations and Conclusions

### Observations:
- Successfully scraped 100 Amazon reviews for iPhone 15 and stored in CSV.
- Preprocessing reduced noise by lowercasing, removing punctuation, stopwords, and lemmatizing.
- Vocabulary size was approximately X words, with common words like "phone", "good" being frequent.
- Feature matrices were highly sparse (>95%), confirming the need for efficient storage.
- TF-IDF highlighted important terms like "camera", "battery" over common words.
- Sentiment classification showed TF-IDF slightly outperforming BoW in some cases due to weighting.

### Conclusions:
- Text feature engineering is crucial for converting unstructured text to numerical features.
- BoW and TF-IDF are foundational but lack semantic understanding; modern embeddings are recommended for advanced tasks.
- Sparse matrices require specialized handling for scalability.
- For sentiment analysis, TF-IDF often provides better performance by emphasizing unique terms.
- Real-world applications should consider domain-specific preprocessing and larger datasets for better results.

### Screenshots:
- [Include screenshots of outputs here, e.g., vocabulary, matrices, model results]